# ANALYSE-04 : Trois primitives de PFR, et l'endroit exact où elles cessent de valoir

**Série** : SymbolicAI / Lean — Digestions de résultats profonds, companion du capstone [Lean-20 — Digestions Tao](../Lean-20-Capstone-Digestions-Tao-Python.ipynb)
**Source** : ANALYSE-03-PFR-Lean.ipynb (digestion PFR par Gowers-Green-Manners-Tao, 2023)
**Lac source** : https://github.com/teorth/pfr (formalisation collaborative Lean 4)
**Kernel** : `rust` (evcxr, `std` seul) — le jumeau Rust mesure les mêmes primitives que le carnet Python, les preuves formelles restent dans ANALYSE-03

**Navigation** : [Index](README.md) | [ANALYSE-03 - PFR Entropy Method <<](ANALYSE-03-PFR-Lean.ipynb) | [ANALYSE-04 Python — source du portage](ANALYSE-04-PFR-Primitives-Python.ipynb) | [ANALYSE-05 - KLS >>](ANALYSE-05-KLS-Lean-Python.ipynb)

***

## Pourquoi ce notebook

La digestion ANALYSE-03 a établi l'essentiel : PFR est un théorème de « compression ⇒ structure », et l'entropie y est **monnaie commune** entre combinatoire additive, machines causales et ICT. Ce qu'ANALYSE-04 ajoute n'est pas une reprise de la preuve — c'est l'**extraction de primitives transportables** et, indissociablement, **le test de leurs limites**.

Trois primitives, et **une seule** est inconditionnellement transportable :

| # | Primitive | Transport |
|---|---|---|
| 1 | Distance entropique de Ruzsa — `d[X;Y] = H(X′−Y′) − ½H(X′) − ½H(Y′)` | **conditionnel** : exige une structure de groupe additif |
| 2 | Décomposition projection / fibres — `X ⟶ π(X) + X\|π(X)` | large : règle de chaîne + information mutuelle conditionnelle |
| 3 | La fonctionnelle τ — descente strictement monotone ; quand elle ne peut plus descendre, **la structure est forcée** | patron, sous hypothèses à expliciter |

## Tester les limites des primitives

> **Une primitive qui ne transfère pas est un résultat.**

La distance de Ruzsa exige une structure de groupe additif : des proxys **sans structure de groupe** ne justifient pas son application. Le notebook cherche donc **où ça casse**, aussi bien que là où ça marche.

## Honnêteté de portée

> PFR ne « prouve » évidemment rien pour ICT — son résultat est spécifique à la combinatoire additive, avec des propriétés particulières de `F₂ⁿ` jusque dans son endgame. **Mais il fournit un exemple CERTIFIÉ où une quantité informationnelle FORCE une structure algébrique. C'est beaucoup plus précieux qu'une analogie.**

Et sur `τ_strictly_decreases` : c'est un **analogue de fonction de Lyapunov pour la preuve**, pas une Lyapunov dynamique au sens physique. La qualification doit figurer dans le notebook.

## Plan

1. Primitive 1 — Distance entropique de Ruzsa : énoncé, calcul, **test de limite**.
2. Primitive 2 — Décomposition projection / fibres : règle de chaîne, **test de limite**.
3. Primitive 3 — La fonctionnelle τ : patron de descente, **test de limite**.
4. Conclusion — lien vers ANALYSE-03 et le lac `pfr`.

**Suite de la sous-série** : [ANALYSE-06b — Sensibilité 2026 (notre lac face aux familles d'openai/math)](ANALYSE-06b-Sensibilite-2026-Lean.ipynb)
**Suite de la sous-série** : [ANALYSE-07 — Percolation critique (notre lac face aux familles 2026 213/236)](ANALYSE-07-Percolation-Critique.ipynb)


## 1. Primitive 1 — Distance entropique de Ruzsa

### 1.1 Énoncé

Soient `X`, `Y` deux variables aléatoires sur un groupe abélien `(G, +)`. On note `X′` une copie indépendante de `X` et `Y′` une copie indépendante de `Y`. La **distance de Ruzsa** est définie par :

```
d[X;Y] = H(X′ − Y′) − ½ H(X′) − ½ H(Y′)
```

où `H` est l'entropie de Shannon en bits.

**Invariants** (sur groupe abélien) :
- `d[X;Y] ≥ 0` toujours (inégalité de conditional mutual information / chaîne).
- `d[X;Y] = d[Y;X]` (symétrique).
- **Translation commune** : `d[X+a; Y+a] = d[X;Y]` pour tout `a` déterministe — l'invariant inconditionnel.
- `d[X;X] = 0` **si et seulement si `X` est uniforme sur un translaté d'un sous-groupe** (en particulier, uniforme sur `G` tout entier — `G` est lui-même le translaté `0 + G`) ; pour une loi quelconque, seule `d[X;X] ≥ 0` est garantie — le test 4 mesure un `d[X;X] = 0.5` bit et le test 5 la nullité sur le translaté `{1,3} = 1 + {0,2}`, qui n'est **pas** un sous-groupe.
- **Triangle** : `d[X;Z] ≤ d[X;Y] + d[Y;Z]`.

**Sémantique** : `d[X;Y]` mesure combien « les distributions sont loin d'être des translatées d'un même sous-groupe uniforme ».

### 1.2 Calcul sur son terrain — un groupe additif fini

On prend `G = ℤ/nℤ` et on compare deux variables `X`, `Y` à distributions connues.


In [2]:
// Helpers de la distance de Ruzsa. Sous evcxr chaque cellule est autonome :
// elle redefinit tout ce qu'elle nomme (fonctions et signatures comprises).
#[allow(dead_code)]
fn entropy_bits(dist: &[(usize, f64)]) -> f64 {
    -dist.iter().filter(|(_, p)| *p > 0.0).map(|(_, p)| p * p.log2()).sum::<f64>() + 0.0
}
// Ordre d'insertion premier-vu : la cle deja vue voit sa valeur AJOUTEE
// (semantique Counter d'une convolution).
fn bump(map: &mut Vec<(usize, f64)>, k: usize, v: f64) {
    for e in map.iter_mut() {
        if e.0 == k { e.1 += v; return; }
    }
    map.push((k, v));
}
fn d_ruzsa(x_dist: &[(usize, f64)], y_dist: &[(usize, f64)], n: usize) -> f64 {
    // X' - Y' : convolution signee modulo n, ordre double boucle x puis y.
    let mut xy_minus: Vec<(usize, f64)> = Vec::new();
    for &(x, px) in x_dist {
        for &(y, py) in y_dist {
            bump(&mut xy_minus, (x + n - y) % n, px * py);
        }
    }
    let h_xy_minus = entropy_bits(&xy_minus);
    let h_x = entropy_bits(x_dist);
    let h_y = entropy_bits(y_dist);
    h_xy_minus - 0.5 * h_x - 0.5 * h_y
}
// Test 1 : X et Y uniformes independants sur Z/8Z
let n = 8usize;
let uniform: Vec<(usize, f64)> = (0..n).map(|k| (k, 1.0 / n as f64)).collect();
println!("Test 1 : X = Y uniformes i.i.d. sur \u{2124}/{}\u{2124}", n);
println!("  d[X;Y] = {:.4}  (attendu ~ 0)", d_ruzsa(&uniform, &uniform, n));
// Test 2 : Y = X + 1 (translation)
let shifted: Vec<(usize, f64)> = (0..n).map(|k| {
    let src = (k + n - 1) % n;
    (k, uniform.iter().find(|e| e.0 == src).map(|e| e.1).unwrap())
}).collect();
println!("Test 2 : Y = X + 1 (translation)");
println!("  d[X;Y] = {:.4}  (attendu 0)", d_ruzsa(&uniform, &shifted, n));
// Test 3 : X uniforme sur Z/8Z, Y degenere en 0 (H(Y) = 0).
// d_ruzsa ne recoit que les lois MARGINALES et convole des copies
// independantes X', Y' : aucun argument de couplage n'entre dans ce calcul.
// Ici X' - Y' = X', donc d[X;Y] = H(X) - 1/2 H(X) = 1/2 H(X) = 1.5 bits.
println!("Test 3 : X uniforme, Y concentr\u{e9} en 0");
let y_degen: Vec<(usize, f64)> = (0..n).map(|k| if k == 0 { (k, 1.0) } else { (k, 0.0) }).collect();
println!("  d[X;Y] = {:.4}  (H(Y)=0, formule d\u{e9}g\u{e9}n\u{e8}re)", d_ruzsa(&uniform, &y_degen, n));
// Test 4 : contre-exemple -- d[X;X] = 0 exige l'uniformite de X sur un
// translaté d'un sous-groupe (le test 5 montre qu'un translaté suffit).
// X = {0: 0.5, 1: 0.5} sur Z/4Z : X' - X'' vaut 0 avec proba 0.5,
// et 1, 3 avec proba 0.25 chacun -> H(X'-X'') = 1.5 bits, H(X) = 1 bit,
// donc d[X;X] = 1.5 - 1 = 0.5 bit, PAS 0.
let x_nu: Vec<(usize, f64)> = vec![(0, 0.5), (1, 0.5)];
println!("Test 4 : X = {{0: 0.5, 1: 0.5}} sur Z/4Z -- loi NON uniforme sur le groupe");
println!("  d[X;X] = {:.4}  (attendu 0.5 bit : {{0,1}} n'est pas un translat\u{e9} de sous-groupe)", d_ruzsa(&x_nu, &x_nu, 4));
// L'invariant inconditionnel est la translation COMMUNE : d[X+a; Y+a] = d[X;Y].
for a in 1..=3usize {
    let x_tr: Vec<(usize, f64)> = x_nu.iter().map(|&(k, p)| ((k + a) % 4, p)).collect();
    println!("  d[X+{}; X+{}] = {:.4}  (translation commune : inchang\u{e9})", a, a, d_ruzsa(&x_tr, &x_tr, 4));
}
// Cas favorable : uniforme sur Z/4Z (le sous-groupe entier G = 0 + G), la nullite est recuperee
let u4: Vec<(usize, f64)> = (0..4).map(|k| (k, 0.25)).collect();
println!("Test 4bis : X uniforme sur Z/4Z -> d[X;X] = {:.4}  (nullit\u{e9} de l'uniforme)", d_ruzsa(&u4, &u4, 4));
// Test 5 : temoin de nullite sur un translaté QUI N'EST PAS un sous-groupe.
// X = {1: 0.5, 3: 0.5} sur Z/4Z : support {1,3} = 1 + {0,2}, translate du
// sous-groupe {0,2} -- il ne contient pas 0, ce n'est donc pas un sous-groupe.
// X' - X'' vaut 0 avec proba 0.5 et 2 avec proba 0.5 -> H(X'-X'') = H(X) = 1 bit,
// donc d[X;X] = 0 : la nullite vaut pour tout translate, pas seulement les sous-groupes.
let x_coset: Vec<(usize, f64)> = vec![(1, 0.5), (3, 0.5)];
println!("Test 5 : X = {{1: 0.5, 3: 0.5}} sur Z/4Z -- uniforme sur le translat\u{e9} 1+{{0,2}}, PAS un sous-groupe");
println!("  d[X;X] = {:.4}  (nullit\u{e9} : uniforme sur un translat\u{e9} de sous-groupe)", d_ruzsa(&x_coset, &x_coset, 4));


Test 1 : X = Y uniformes i.i.d. sur ℤ/8ℤ


  d[X;Y] = 0.0000  (attendu ~ 0)


Test 2 : Y = X + 1 (translation)


  d[X;Y] = 0.0000  (attendu 0)


Test 3 : X uniforme, Y concentré en 0


  d[X;Y] = 1.5000  (H(Y)=0, formule dégénère)


... (4 outputs left out)

  d[X+3; X+3] = 0.5000  (translation commune : inchangé)


Test 4bis : X uniforme sur Z/4Z -> d[X;X] = 0.0000  (nullité de l'uniforme)


Test 5 : X = {1: 0.5, 3: 0.5} sur Z/4Z -- uniforme sur le translaté 1+{0,2}, PAS un sous-groupe


  d[X;X] = 0.0000  (nullité : uniforme sur un translaté de sous-groupe)


### Lecture du résultat

Les cinq tests illustrent les **invariants** sur groupe additif :
- Test 1 : variables i.i.d. **uniformes sur le même groupe** ⇒ `d[X;Y] ≈ 0` — la nullité tient à l'uniformité, pas à la seule identité des lois.
- Test 2 : translation d'une loi **uniforme** ⇒ `d[X;Y] = 0` exactement (la distance de Ruzsa est invariante par translation, c'est sa propriété fondamentale).
- Test 3 : dégénérescence — la formule reste définie mais perd sa sémantique ; `d_ruzsa` ne voit que les lois marginales, jamais un couplage.
- Test 4 : **contre-exemple** — `X = {0: ½, 1: ½}` sur `ℤ/4ℤ` n'est pas uniforme sur le groupe, et `d[X;X] = 0.5` bit : l'invariant `d[X;X] = 0` ne vaut que pour `X` uniforme sur un **translaté de sous-groupe**. L'invariant inconditionnel est la translation **commune** `d[X+a; Y+a] = d[X;Y]`, vérifiée au même test ; une translation simple `X = a + Y` donne `d[X;Y] = d[Y;Y]`, nul seulement si `Y` est uniforme sur un **translaté de sous-groupe**.
- Test 5 : **témoin de nullité sur un translaté** — `X = {1: ½, 3: ½}` est uniforme sur le translaté `1 + {0,2}` (qui ne contient pas `0`, donc pas un sous-groupe) et `d[X;X] = 0` : le critère est bien l'uniformité sur un translaté, pas sur un sous-groupe seul. Le contraste (a) de la cellule suivante le rejoint : le même support `{0,1}` est un sous-groupe du groupe de Klein (`d = 0`) mais n'est le translaté d'aucun sous-groupe de `ℤ/4ℤ` (`d = 0.5`).

**Point clé** : la définition **exige** la convolution `X′ − Y′`, qui n'a de sens que sur groupe additif. C'est ce que « Ruzsa exige une structure de groupe additif » veut dire.

### 1.3 Test de limite — quand la primitive ne transporte plus

On garde la formule, les distributions et l'entropie **inchangées**, et on ne fait
varier que l'opération `x − y`. Deux exhibitions :

- **(a)** le même ensemble `{0,1,2,3}` muni de **deux lois de groupe différentes**
  (`ℤ/4ℤ` et le groupe de Klein) donne **deux valeurs différentes** de `d` pour les
  **mêmes** distributions — la valeur de `d` n'est donc pas une fonction des seules
  distributions, mais du couple (distributions, loi de groupe) ;
- **(b)** en retirant l'inversibilité de `x ↦ x − y`, `d` devient **négatif** — et une
  distance négative n'en est plus une.


In [3]:
// 1.3 — Test de limite : ce que la formule cesse de mesurer sans loi de groupe.
// On ne change QUE l'operation « x - y ». Le reste — les distributions, la
// formule, l'entropie — est identique a la cellule precedente.
#[allow(dead_code)]
fn entropy_bits(dist: &[(usize, f64)]) -> f64 {
    -dist.iter().filter(|(_, p)| *p > 0.0).map(|(_, p)| p * p.log2()).sum::<f64>() + 0.0
}
fn bump(map: &mut Vec<(usize, f64)>, k: usize, v: f64) {
    for e in map.iter_mut() {
        if e.0 == k { e.1 += v; return; }
    }
    map.push((k, v));
}
// Semantique dict-comprehension : la cle deja vue est REECRITE, pas sommee.
fn set_first_seen(map: &mut Vec<(usize, f64)>, k: usize, v: f64) {
    for e in map.iter_mut() {
        if e.0 == k { e.1 = v; return; }
    }
    map.push((k, v));
}
fn d_ruzsa_op(x_dist: &[(usize, f64)], y_dist: &[(usize, f64)], sub: &dyn Fn(usize, usize) -> usize) -> f64 {
    let mut xy: Vec<(usize, f64)> = Vec::new();
    for &(x, px) in x_dist {
        for &(y, py) in y_dist {
            bump(&mut xy, sub(x, y), px * py);
        }
    }
    entropy_bits(&xy) - 0.5 * entropy_bits(x_dist) - 0.5 * entropy_bits(y_dist)
}

// Quatre lois « x - y » candidates. Des fn (pas des closures) : evcxr
// persiste les fonctions, et une fn se coerce en &dyn Fn la ou il le faut.
fn z4(a: usize, b: usize) -> usize { (a + 4 - b) % 4 }   // Z/4Z : groupe cyclique
fn klein(a: usize, b: usize) -> usize { a ^ b }          // (Z/2Z)^2 : groupe de Klein
fn sub_max(a: usize, b: usize) -> usize { a.saturating_sub(b) }  // non inversible
fn sub_mul(a: usize, b: usize) -> usize { (a * b) % 4 }   // non inversible

// (a) Meme ensemble, memes distributions, deux lois de groupe differentes.
//     X = Y = uniforme sur {0, 1}, plongees dans un ensemble a 4 elements.
let x: Vec<(usize, f64)> = vec![(0, 0.5), (1, 0.5)];
let y = x.clone();

println!("(a) m\u{ea}mes distributions, deux lois de groupe sur le m\u{ea}me ensemble {{0,1,2,3}}");
let mut ecrasement: Vec<(usize, f64)> = Vec::new();
for xk in [0usize, 1] {
    for yk in [0usize, 1] {
        set_first_seen(&mut ecrasement, z4(xk, yk), 0.25);
    }
}
println!("  \u{2124}/4\u{2124}  : H(X\u{2212}Y) = {:.4} bits   \u{2192}  d[X;Y] = {:.4}", entropy_bits(&ecrasement), d_ruzsa_op(&x, &y, &z4));
println!("  Klein : d[X;Y] = {:.4}", d_ruzsa_op(&x, &y, &klein));
println!("  La valeur de d n'est donc PAS une fonction des seules distributions :");
println!("  c'est une fonction du couple (distributions, loi de groupe).");

// (b) On retire l'inversibilite : « x - y » cesse d'etre une bijection en x.
//     Si d devient negatif, ce n'est plus une distance.
// MT19937 de CPython (init_by_array avec l'indice de cle j), recopie tel quel
// des Ecueils : random() = 53 bits, gammavariate(1.0, 1.0) = -log(1-random()).
struct Mt19937 { mt: [u32; 624], idx: usize }
impl Mt19937 {
    fn init_genrand(s: u32) -> [u32; 624] {
        let mut mt = [0u32; 624];
        mt[0] = s;
        for i in 1..624 {
            mt[i] = 1812433253u32.wrapping_mul(mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_add(i as u32);
        }
        mt
    }
    fn new(seed: u32) -> Mt19937 { Self::new_by_array(&[seed]) }  // random.Random(7)
    fn new_by_array(key: &[u32]) -> Mt19937 {
        let mut mt = Self::init_genrand(19650218);
        let (mut i, mut j) = (1usize, 0usize);
        for _ in 0..624.max(key.len()) {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1664525))
                .wrapping_add(key[j]).wrapping_add(j as u32);
            i += 1; j += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
            if j >= key.len() { j = 0; }
        }
        for _ in 0..623 {
            mt[i] = (mt[i] ^ (mt[i - 1] ^ (mt[i - 1] >> 30)).wrapping_mul(1566083941))
                .wrapping_sub(i as u32);
            i += 1;
            if i >= 624 { mt[0] = mt[623]; i = 1; }
        }
        mt[0] = 0x8000_0000;
        Mt19937 { mt, idx: 624 }
    }
    fn next_u32(&mut self) -> u32 {
        if self.idx >= 624 {
            for k in 0..624 {
                let y = (self.mt[k] & 0x8000_0000) | (self.mt[(k + 1) % 624] & 0x7fff_ffff);
                let mut v = self.mt[(k + 397) % 624] ^ (y >> 1);
                if y & 1 != 0 { v ^= 0x9908_b0df; }
                self.mt[k] = v;
            }
            self.idx = 0;
        }
        let mut y = self.mt[self.idx];
        self.idx += 1;
        y ^= y >> 11;
        y ^= (y << 7) & 0x9d2c_5680;
        y ^= (y << 15) & 0xefc6_0000;
        y ^= y >> 18;
        y
    }
    fn random(&mut self) -> f64 {              // random.random() : 53 bits
        let a = (self.next_u32() >> 5) as f64;
        let b = (self.next_u32() >> 6) as f64;
        (a * 67108864.0 + b) / 9007199254740992.0
    }
    fn gammavariate_1_1(&mut self) -> f64 {    // CPython 3.14.3, branche alpha == 1.0
        -((1.0 - self.random()).ln())          // -log(1.0 - random()) * 1.0
    }
}

fn dirichlet(k: usize, rng: &mut Mt19937) -> Vec<(usize, f64)> {
    let mut g = Vec::new();
    for _ in 0..k { g.push(rng.gammavariate_1_1()); }
    let s = g.iter().fold(0.0, |acc, v| acc + v);
    (0..k).map(|i| (i, g[i] / s)).collect()
}

// Bloc borne : le tirage 40 000 n'a rien a persister au-dela de la cellule.
{
    let lois: [(&str, &str, &dyn Fn(usize, usize) -> usize); 4] = [
        ("(a \u{2212} b) mod 4", "groupe cyclique", &z4),
        ("a XOR b",       "groupe de Klein", &klein),
        ("max(a \u{2212} b, 0)", "non inversible",  &sub_max),
        ("(a \u{d7} b) mod 4", "non inversible",  &sub_mul),
    ];
    let mut rng = Mt19937::new(7);
    println!("\n(b) 40 000 couples de distributions tir\u{e9}s au hasard sur {{0,1,2,3}} :");
    for &(nom, nature, sub) in &lois {
        let mut d_min = f64::INFINITY;
        for _ in 0..40_000 {
            let dx = dirichlet(4, &mut rng);
            let dy = dirichlet(4, &mut rng);
            let d = d_ruzsa_op(&dx, &dy, sub);
            if d < d_min { d_min = d; }
        }
        let verdict = if d_min > -1e-9 { "d \u{2265} 0 : distance" } else { "d < 0 : PLUS une distance" };
        println!("  {:<14} {:<16} \u{2192}  min d = {:+.4}   {}", nom, nature, d_min, verdict);
    }
}
println!("\n  Les deux op\u{e9}rations non inversibles rendent d n\u{e9}gatif. Une \u{ab} distance \u{bb}");
println!("  n\u{e9}gative n'en est pas une : la primitive a cess\u{e9} de valoir, et on voit o\u{f9}.");


(a) mêmes distributions, deux lois de groupe sur le même ensemble {0,1,2,3}


  ℤ/4ℤ  : H(X−Y) = 1.5000 bits   →  d[X;Y] = 0.5000


  Klein : d[X;Y] = 0.0000


  La valeur de d n'est donc PAS une fonction des seules distributions :


  c'est une fonction du couple (distributions, loi de groupe).


(b) 40 000 couples de distributions tirés au hasard sur {0,1,2,3} :


... (2 outputs left out)

  max(a − b, 0)  non inversible   →  min d = -1.2699   d < 0 : PLUS une distance


  (a × b) mod 4  non inversible   →  min d = -0.9756   d < 0 : PLUS une distance


  Les deux opérations non inversibles rendent d négatif. Une « distance »


  négative n'en est pas une : la primitive a cessé de valoir, et on voit où.


### Verdict Primitive 1

**Transportable sous condition** : `d_ruzsa[X;Y]` exige une **structure de groupe additif** `(G, +)`. La cellule précédente le montre en deux temps — la valeur de `d` **change** quand on change la loi de groupe à distributions fixées, et sans inversibilité de `x ↦ x − y` elle **passe sous zéro**, donc cesse d'être une distance. C'est exactement la classe « conditionnelle » du tableau d'introduction.

**Leçon** : la primitive a une **puissance incomparable** sur son terrain (groupes abéliens, `F₂ⁿ`, entropie appliquée à la combinatoire additive), mais l'**utiliser hors de ce terrain** serait du « mapping affine » — c'est-à-dire plaquer une structure sans la mériter. Le carnet ANALYSE-03 lui-même s'en garde : la preuve PFR ne s'applique qu'à `F₂ⁿ`, pas à un objet métrique général.


### 1.4 Test 6 — l'inégalité triangulaire, mesurée

L'énoncé de 1.1 liste le **triangle** `d[X;Z] ≤ d[X;Y] + d[Y;Z]` parmi les invariants — c'est le seul de la liste qui n'était pas encore mesuré, alors qu'il n'est pas une évidence de la formule : l'inégalité triangulaire entropique de Ruzsa est un théorème du terrain, démontré par Tao, *Sumset and inverse sumset theory for Shannon entropy* (arXiv:0906.4387), théorème 1.10, équation (16), sous la forme `distR(X, Z) ≤ distR(X, Y) + distR(Y, Z)`. Le test suivant la mesure sur trois triplets de `ℤ/8ℤ` choisis pour discriminer — égalité exacte, saturation, triplet tendu — en réutilisant `d_ruzsa` telle quelle, sans réimplémentation.

In [4]:
// 1.4 — Test 6 : l'inegalite triangulaire d[X;Z] <= d[X;Y] + d[Y;Z], mesuree.
// d_ruzsa (cellule 1.2) est reutilisee telle quelle : trois distances par
// triplet, l'inegalite lue sur trois temoins qui discriminent.
#[allow(dead_code)]
fn entropy_bits(dist: &[(usize, f64)]) -> f64 {
    -dist.iter().filter(|(_, p)| *p > 0.0).map(|(_, p)| p * p.log2()).sum::<f64>() + 0.0
}
fn bump(map: &mut Vec<(usize, f64)>, k: usize, v: f64) {
    for e in map.iter_mut() {
        if e.0 == k { e.1 += v; return; }
    }
    map.push((k, v));
}
fn d_ruzsa(x_dist: &[(usize, f64)], y_dist: &[(usize, f64)], n: usize) -> f64 {
    let mut xy_minus: Vec<(usize, f64)> = Vec::new();
    for &(x, px) in x_dist {
        for &(y, py) in y_dist {
            bump(&mut xy_minus, (x + n - y) % n, px * py);
        }
    }
    entropy_bits(&xy_minus) - 0.5 * entropy_bits(x_dist) - 0.5 * entropy_bits(y_dist)
}
fn lit_triplet(nom: &str, x: &[(usize, f64)], y: &[(usize, f64)], z: &[(usize, f64)], n: usize) {
    let (dxy, dyz, dxz) = (d_ruzsa(x, y, n), d_ruzsa(y, z, n), d_ruzsa(x, z, n));
    let (borne, marge) = (dxy + dyz, dxy + dyz - dxz);
    println!("{}", nom);
    println!("  d[X;Y] = {:.4}   d[Y;Z] = {:.4}   d[X;Z] = {:.4}", dxy, dyz, dxz);
    println!("  in\u{e9}galit\u{e9} : {:.4} <= {:.4}   marge = {:+.4} bit   {}", dxz, borne, marge,
             if dxz <= borne + 1e-9 { "OK" } else { "VIOLEE" });
}
let n8 = 8usize;
let a: Vec<(usize, f64)> = vec![(0, 0.25), (2, 0.25), (4, 0.25), (6, 0.25)];       // uniforme sur le sous-groupe A = {0,2,4,6}
let a_plus: Vec<(usize, f64)> = vec![(1, 0.25), (3, 0.25), (5, 0.25), (7, 0.25)];  // uniforme sur le translaté A + 1
let u8: Vec<(usize, f64)> = (0..n8).map(|k| (k, 1.0 / n8 as f64)).collect();       // uniforme sur Z/8Z tout entier
let degen0: Vec<(usize, f64)> = (0..n8).map(|k| if k == 0 { (k, 1.0) } else { (k, 0.0) }).collect();  // degenere en 0

println!("Test 6 : l'in\u{e9}galit\u{e9} triangulaire d[X;Z] <= d[X;Y] + d[Y;Z], sur Z/8Z");
println!();
lit_triplet("Triplet 1 -- \u{e9}galit\u{e9} exacte : X = Y = Z uniformes sur A = {0,2,4,6}", &a, &a, &a, n8);
println!();
lit_triplet("Triplet 2 -- saturation sans colin\u{e9}arit\u{e9} : X, Y sur A ; Z sur A + 1", &a, &a, &a_plus, n8);
println!();
lit_triplet("Triplet 3 -- tendu : Y d\u{e9}g\u{e9}n\u{e9}r\u{e9} en 0, X uniforme sur Z/8Z, Z sur A", &u8, &degen0, &a, n8);


Test 6 : l'inégalité triangulaire d[X;Z] <= d[X;Y] + d[Y;Z], sur Z/8Z


Triplet 1 -- égalité exacte : X = Y = Z uniformes sur A = {0,2,4,6}


  d[X;Y] = 0.0000   d[Y;Z] = 0.0000   d[X;Z] = 0.0000


  inégalité : 0.0000 <= 0.0000   marge = +0.0000 bit   OK


Triplet 2 -- saturation sans colinéarité : X, Y sur A ; Z sur A + 1


  d[X;Y] = 0.0000   d[Y;Z] = 0.0000   d[X;Z] = 0.0000


  inégalité : 0.0000 <= 0.0000   marge = +0.0000 bit   OK


Triplet 3 -- tendu : Y dégénéré en 0, X uniforme sur Z/8Z, Z sur A


  d[X;Y] = 1.5000   d[Y;Z] = 1.0000   d[X;Z] = 0.5000


  inégalité : 0.5000 <= 2.5000   marge = +2.0000 bit   OK


### Lecture du résultat — Test 6

- **Triplet 1** (`X = Y = Z` uniformes sur le sous-groupe `A = {0,2,4,6}`) : les trois distances valent `0` — l'inégalité est une **égalité exacte** (`0 ≤ 0 + 0`).
- **Triplet 2** (`X`, `Y` sur `A` ; `Z` sur le translaté `A + 1`) : les trois distances valent encore `0` alors que `X` et `Z` n'ont **pas la même loi** — l'inégalité est **saturée sans colinéarité** des lois. C'est le phénomène du test 2 : deux uniformes sur des translatés d'un même sous-groupe sont à distance nulle, et la borne ne peut pas descendre plus bas.
- **Triplet 3** (`Y` dégénéré en `0` entre `X` uniforme sur `ℤ/8ℤ` et `Z` uniforme sur `A`) : `d[X;Y] = ½H(X) = 1.5` bit et `d[Y;Z] = ½H(Z) = 1.0` bit — la borne vaut `2.5` bits, et la mesure rend `d[X;Z] = 0.5` bit : l'inégalité tient avec une **marge de 2.0 bits**. Un test qui ne rendrait que `0.5 ≤ 2.5` n'apprendrait rien ; les trois témoins ensemble disent où l'inégalité colle et où elle respire.

Le contrôle négatif est déjà dans le carnet : le test de limite (b) de 1.3 montre que sans inversibilité de `x ↦ x − y`, `d` devient **négatif** — la triangularité meurt avec la structure de groupe, comme les autres invariants. La ligne d'énoncé du triangle en 1.1 est désormais adossée à sa mesure.

### Exercice 1 — Ruzsa sur deux groupes de même ordre : Z/4Z contre (Z/2Z)²

La cellule du cours calcule `d[X;Y]` sur Z/8Z (`n = 8`). Le verdict ci-dessus dit que la valeur de `d` **change quand on change la loi de groupe, à distributions fixées** — vérifiez-le sur deux groupes d'ordre 4 construits sur les mêmes étiquettes {0, 1, 2, 3} :

- **Z/4Z** (cyclique) : `add_z4(x, y) = (x + y) % 4` ;
- **(Z/2Z)²** (groupe de Klein) : `add_klein(x, y) = x ^ y` (ou exclusif bit à bit sur les étiquettes codées en 2 bits).

Pour les **mêmes** distributions — X uniforme sur {0, 1} et Y uniforme sur {2, 3} — calculez `d[X;Y] = H(X−Y) − ½H(X) − ½H(Y)` sous **chacune des deux lois**, et comparez les deux valeurs.

**Indice** : dans (Z/2Z)² chaque élément est son propre inverse, donc `X−Y = X+Y` ; côté Z/4Z, l'inverse de `y` est `(4−y) % 4`. Écrivez la convolution **une seule fois**, paramétrée par la loi `add`.
- **Etape 1** : écrire `convolution(px, py, add)` — le produit de convolution sous une loi de groupe quelconque.
- **Etape 2** : composer `d` en réutilisant `entropy_bits` de la cellule du cours.
- **Etape 3** : afficher les deux valeurs et conclure : la distance dépend de la loi de groupe, pas seulement des distributions.


In [5]:
// Exercice 1 -- d[X;Y] sur Z/4Z puis (Z/2Z)^2, memes etiquettes {0,1,2,3} (a completer)
#[allow(dead_code)]
fn convolution(px: &[(usize, f64)], py: &[(usize, f64)], add: &dyn Fn(usize, usize) -> usize) -> Vec<(usize, f64)> {
    // Produit de convolution de deux lois sous une loi de groupe add(x, y).
    Vec::new() // TODO etudiant (Etape 1)
}
#[allow(dead_code)]
fn d_ruzsa_groupe(px: &[(usize, f64)], py: &[(usize, f64)], add: &dyn Fn(usize, usize) -> usize, inverse: &dyn Fn(usize) -> usize) -> f64 {
    // d[X;Y] = H(X-Y) - 0.5*H(X) - 0.5*H(Y) ; X-Y se calcule en convoluant X et -Y = inverse(Y).
    0.0 // TODO etudiant (Etape 2)
}
// Etape 3 : X uniforme sur {0,1}, Y uniforme sur {2,3}, deux lois de groupe -- comparer.
println!("Exercice a completer : d[X;Y] sous Z/4Z puis sous (Z/2Z)^2, memes X et Y.");
println!("Attendu : deux valeurs distinctes -- la distance depend de la loi de groupe.");


Exercice a completer : d[X;Y] sous Z/4Z puis sous (Z/2Z)^2, memes X et Y.


Attendu : deux valeurs distinctes -- la distance depend de la loi de groupe.


## 2. Primitive 2 — Décomposition projection / fibres

### 2.1 Énoncé

Soient `X` une variable aléatoire et `Y` une variable auxiliaire. On note
`π(X) := 𝔼[X | Y]` (espérance conditionnelle L2) ou plus généralement une
**projection** sur une σ-algèbre. La **décomposition informationnelle** est :

```
X = π(X) + (X − π(X))
```

où `X − π(X)` désigne le **résidu** (composante inexpliquée par la
projection). Cette écriture est un raccourci de présentation : la
primitive est **informationnelle** (entropies), pas numérique — sur des
v.a. continues le « + » n'a pas de sens au sens usuel.

**Invariants** (règle de chaîne) :

- `H(X) = H(π(X)) + H(X | π(X))` — la chaîne informationnelle, universelle.
- `I(X ; Y) = H(X) − H(X | Y)` — définition de l'information mutuelle.
- `𝔼[X − M | M] = 0` — **le résidu est d'espérance conditionnelle nulle
  par rapport à la projection L2** `M := 𝔼[X | Y]` (régression). C'est la
  *définition* de l'espérance conditionnelle, et la seule à la garantir.
  On note `P = g(X)` une projection générale (pas L2) : `P` n'a aucune
  raison de satisfaire `𝔼[X − P | P] = 0`. Le Cas 2 du carnet utilise
  `P = 𝟙{X ≥ 15}` (indicatrice binaire), et l'empirie donne
  `𝔼[X − P | P = high] ≈ 16.0` et `𝔼[X − P | P = low] ≈ 11.93` : l'écart
  (~4 points) est précisément ce que la projection binaire ignore de la
  note. Cette propriété est strictement plus faible que l'indépendance :
  pour `M` L2, l'orthogonalité implique `Cov(X − M, M) = 0`
  (linéarité de l'espérance conditionnelle) ; pour `P` général, ni
  `Cov(X − P, P) = 0` (Cas 2 : 0.98 ≠ 0) ni `I(X − P ; P) = 0`
  (Cas 2 : 0.78 bit) ne tiennent. `I(X − π(X) ; π(X)) = 0` n'est pas
  universel : il suppose un modèle gaussien, où la décorrélation
  implique l'indépendance. La primitive 2 informationnelle n'a donc
  PAS le statut de définition universelle qu'on lui prête parfois :
  `I(X − π ; π) = 0` est un cas particulier, pas l'identité générique.
- `I(X ; π(X)) = H(π(X))` — et **non 0** en général : `π(X)` est une
  fonction déterministe de `X`, donc toute l'information de `π(X)` vient
  de `X`.

**Erreur à éviter** : l'affirmation `I(X ; π(X)) = 0` est **fausse** dès
que `π` est une fonction déterministe de `X` (le cas usuel). La quantité
qui s'annule **dans le cas gaussien** est `I(X − π(X) ; π(X)) = 0` (le
résidu est indépendant de la projection), **pas** `I(X ; π(X))`. En
dehors du gaussien, l'invariant universellement vrai est
`𝔼[X − π(X) | π(X)] = 0`, qui ne donne qu'une *décorrélation*, pas une
indépendance — mesuré sur les 25 paires du carnet, pour la projection
indicatrice `π(X) = 𝟙{X≥15}` (Cas 2), on trouve `I(X − π(X) ; π(X)) ≈
0.78 bit` et `Cov(X − π(X), π(X)) ≈ 0.98` (≠ 0) : le résidu n'est
même pas décorrélé, l'orthogonalité n'étant garantie que pour la
projection L2 `π(X) = 𝔼[X | ·]`.

### 2.2 Application : décomposer un dataset ML



In [6]:
// Primitive 2 — Décomposition projection / fibres sur un cas concret
//
// On considère deux scénarios, qui testent des identités DIFFÉRENTES :
//   Cas 1 : projection sur Y (cours). Y n'est PAS une fonction de X,
//           donc on ne peut pas écrire H(X) = H(π(X)) + H(X | π(X))
//           avec π(X) := Y. La chaîne correcte est H(X) = I(X;Y) + H(X|Y),
//           et l'écart H(Y) − I(X;Y) > 0 mesure l'information que Y porte
//           sur autre chose que X.
//   Cas 2 : projection π(X) = 𝟙{X ≥ 15}. π EST une fonction de X,
//           donc on peut écrire H(X) = H(π(X)) + H(X | π(X)) ET
//           I(X ; π(X)) = H(π(X)) ≠ 0 (la cellule 8 disait I = 0 : faux).
#[allow(dead_code)]
fn entropy_bits<K: Copy + PartialEq>(dist: &[(K, f64)]) -> f64 {
    -dist.iter().filter(|(_, p)| *p > 0.0).map(|(_, p)| p * p.log2()).sum::<f64>() + 0.0
}
// Compteur a ordre d'insertion premier-vu (semantique Counter de Python).
fn bump_count<K: Copy + PartialEq>(map: &mut Vec<(K, i64)>, k: K) {
    for e in map.iter_mut() {
        if e.0 == k { e.1 += 1; return; }
    }
    map.push((k, 1));
}
fn normalize<K: Copy + PartialEq>(counts: &[(K, i64)]) -> Vec<(K, f64)> {
    let total = counts.iter().map(|e| e.1).sum::<i64>();
    counts.iter().map(|&(k, c)| (k, c as f64 / total as f64)).collect()
}
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

let notes_brutes: [(&str, i64); 25] = [
    ("Math", 14), ("Math", 16), ("Math", 12), ("Math", 18), ("Math", 15),
    ("Phys", 11), ("Phys", 13), ("Phys", 9),  ("Phys", 12), ("Phys", 14),
    ("Info", 17), ("Info", 19), ("Info", 16), ("Info", 18), ("Info", 20),
    ("Bio",  10), ("Bio",  12), ("Bio",  11), ("Bio",  13), ("Bio",  9),
    ("Chim", 13), ("Chim", 15), ("Chim", 14), ("Chim", 16), ("Chim", 12),
];
let n_total = notes_brutes.len();
let mut notes_counter: Vec<(i64, i64)> = Vec::new();
for &(_, n) in &notes_brutes { bump_count(&mut notes_counter, n); }
let h_x = entropy_bits(&normalize(&notes_counter));

// === Cas 1 — projection sur Y (cours). Y n'est PAS fonction de X. ===
let mut by_cours: Vec<(&str, Vec<i64>)> = Vec::new();
for &(c, n) in &notes_brutes {
    if let Some(e) = by_cours.iter_mut().find(|e| e.0 == c) { e.1.push(n); } else { by_cours.push((c, vec![n])); }
}
let mut h_x_given_y = 0.0f64;
for ns in &by_cours {
    let p_y = ns.1.len() as f64 / n_total as f64;
    let mut ctr: Vec<(i64, i64)> = Vec::new();
    for &v in &ns.1 { bump_count(&mut ctr, v); }
    let h_n_given_c = entropy_bits(&normalize(&ctr));
    h_x_given_y += p_y * h_n_given_c;
}
let mut cours_counter: Vec<(&str, i64)> = Vec::new();
for &(c, _) in &notes_brutes { bump_count(&mut cours_counter, c); }
let h_y = entropy_bits(&normalize(&cours_counter));
let i_xy = h_x - h_x_given_y;
println!("=== Cas 1 : projection sur Y (cours). Y n'est PAS fonction de X. ===");
println!("  H(X)        = {:.3} bits", h_x);
println!("  H(X | Y)    = {:.3} bits  (variation inexpliqu\u{e9}e par le cours)", h_x_given_y);
println!("  H(Y)        = {:.3} bits", h_y);
println!("  I(X;Y)      = H(X) - H(X|Y) = {:.3} bits  (info que le cours porte sur la note)", i_xy);
println!("  \u{c9}cart H(Y) \u{2212} I(X;Y) = {:.3} bits", h_y - i_xy);
println!("    \u{2192} I(X;Y) = {:.3} \u{2260} H(Y) = {:.3} : Y porte {:.3} bits", i_xy, h_y, h_y - i_xy);
println!("    d'information qui ne concernent pas X (t\u{e9}moin num\u{e9}rique de l'\u{e9}cart).");
println!("    C'est pr\u{e9}cis\u{e9}ment pour \u{e7}a qu'on ne peut PAS \u{e9}crire");
println!("    H(X) = H(Y) + H(X|Y) : ce serait supposer Y = \u{3c0}(X) (fonction de X).");
println!("  Cha\u{ee}ne v\u{e9}rifi\u{e9}e : H(X) = {:.3} \u{2248} H(X|Y) + I(X;Y) = {:.3}", h_x, h_x_given_y + i_xy);
println!();

// === Cas 2 — projection π(X) = 𝟙{X ≥ 15}. π est fonction de X. ===
let seuil = 15i64;
let notes_projetees: Vec<&str> = notes_brutes.iter().map(|&(_, n)| if n >= seuil { "high" } else { "low" }).collect();
let mut pix_counter: Vec<(&str, i64)> = Vec::new();
for &p in &notes_projetees { bump_count(&mut pix_counter, p); }
let pi_x_dist = normalize(&pix_counter);
let h_pi_x = entropy_bits(&pi_x_dist);
let mut by_pi: Vec<(&str, Vec<i64>)> = Vec::new();
for ((_, n), proj) in notes_brutes.iter().zip(&notes_projetees) {
    if let Some(e) = by_pi.iter_mut().find(|e| e.0 == *proj) { e.1.push(*n); } else { by_pi.push((proj, vec![*n])); }
}
let mut h_x_given_pi_x = 0.0f64;
for ns in &by_pi {
    let p_pi = ns.1.len() as f64 / n_total as f64;
    let mut ctr: Vec<(i64, i64)> = Vec::new();
    for &v in &ns.1 { bump_count(&mut ctr, v); }
    h_x_given_pi_x += p_pi * entropy_bits(&normalize(&ctr));
}
let i_x_pi_x = h_x - h_x_given_pi_x;
println!("=== Cas 2 : projection \u{3c0}(X) = \u{1d7d9}{{X \u{2265} {}}}. \u{3c0} est fonction de X. ===", seuil);
println!("  H(\u{3c0}(X))     = {:.3} bits  (entropie de l'indicatrice 'note \u{2265} {}')", h_pi_x, seuil);
println!("  H(X | \u{3c0}(X)) = {:.3} bits  (variation intra-fibre)", h_x_given_pi_x);
println!("  I(X ; \u{3c0}(X)) = H(X) - H(X | \u{3c0}(X)) = {:.3} bits", i_x_pi_x);
println!("  I(X ; \u{3c0}(X)) = H(\u{3c0}(X)) ? {}  \u{2192}  I(X;\u{3c0}(X)) = {:.3}", py_bool((i_x_pi_x - h_pi_x).abs() < 1e-9), i_x_pi_x);
println!("    \u{2262} 0 bits (la cellule 8 affirmait I = 0 : c'est FAUX).");
println!("    Ici \u{3c0}(X) est une fonction de X, donc l'identit\u{e9} correcte est");
println!("    I(X ; \u{3c0}(X)) = H(\u{3c0}(X)) : la projection porte TOUTE son information");
println!("    de X, et l'\u{e9}cart \u{e0} 0 mesure exactement le contenu informationnel de \u{3c0}.");
println!("  Cha\u{ee}ne v\u{e9}rifi\u{e9}e : H(X) = {:.3} \u{2248} H(\u{3c0}(X)) + H(X | \u{3c0}(X)) = {:.3}", h_x, h_pi_x + h_x_given_pi_x);
// === Témoins directs cités en cellule 8 (C.5 prose quantitative, ancrage) ===
// Résidu r = X − π_num(X) où π_num = 1 si X ≥ 15, 0 sinon. La prose
// des cellules 8 et 10 cite I(X−π(X); π(X)) ≈ 0.78 bit et Cov ≈ 0.98
// — on les calcule ici sur les 25 paires du carnet, qui sont la base
// de toutes les autres mesures (donc falsifiables par le lecteur).
let residus: Vec<i64> = notes_brutes.iter().zip(&notes_projetees)
    .map(|(&(_, n), &p)| n - if p == "high" { 1 } else { 0 }).collect();
let pi_num: Vec<i64> = notes_projetees.iter().map(|&p| if p == "high" { 1 } else { 0 }).collect();
let mut res_ctr: Vec<(i64, i64)> = Vec::new();
for &r in &residus { bump_count(&mut res_ctr, r); }
let h_res = entropy_bits(&normalize(&res_ctr));
let mut respi_ctr: Vec<((i64, &str), i64)> = Vec::new();
for (&r, &p) in residus.iter().zip(&notes_projetees) { bump_count(&mut respi_ctr, (r, p)); }
let h_res_pi = entropy_bits(&normalize(&respi_ctr));
let i_res_pi = h_res + h_pi_x - h_res_pi;
let mu_r = residus.iter().sum::<i64>() as f64 / n_total as f64;
let mu_p = pi_num.iter().sum::<i64>() as f64 / n_total as f64;
let cov_res_pi = residus.iter().zip(&pi_num)
    .map(|(&r, &p)| (r as f64 - mu_r) * (p as f64 - mu_p)).sum::<f64>() / n_total as f64;
println!("  I(X\u{2212}\u{3c0}(X); \u{3c0}(X)) = {:.3} bits  (t\u{e9}moin C.5 cit\u{e9} cellule 8)", i_res_pi);
println!("  Cov(X\u{2212}\u{3c0}(X), \u{3c0}(X)) = {:.3}  (t\u{e9}moin C.5 cit\u{e9} cellule 8)", cov_res_pi);
// === Témoin E[X−π(X) | π(X)] = 0 ? UNIQUEMENT pour la projection L2 M = 𝔼[X|·] ===
let resi_high: Vec<i64> = residus.iter().zip(&pi_num).filter(|&(_, &p)| p == 1).map(|(&r, _)| r).collect();
let resi_low: Vec<i64> = residus.iter().zip(&pi_num).filter(|&(_, &p)| p == 0).map(|(&r, _)| r).collect();
let e_res_given_high = resi_high.iter().sum::<i64>() as f64 / resi_high.len() as f64;
let e_res_given_low = resi_low.iter().sum::<i64>() as f64 / resi_low.len() as f64;
println!("  E[X\u{2212}\u{3c0}(X) | \u{3c0}(X)=high] = {:.3}  (t\u{e9}moin : \u{2260} 0)", e_res_given_high);
println!("  E[X\u{2212}\u{3c0}(X) | \u{3c0}(X)=low]  = {:.3}  (t\u{e9}moin : \u{2260} 0)", e_res_given_low);
println!("  \u{2192} \u{3c0} = \u{1d7d9}{{X\u{2265}{}}} n'est PAS la projection L2 M = \u{1d53c}[X|\u{b7}] :", seuil);
println!("    l'orthogonalit\u{e9} \u{1d53c}[X\u{2212}\u{3c0}|\u{3c0}]=0 ne tient qu'au sens L2 (r\u{e9}gression),");
println!("    pas pour une projection g\u{e9}n\u{e9}rale. Identit\u{e9} universelle :");
println!("    \u{1d53c}[X\u{2212}M | M] = 0 (M = projection L2) ; pas \u{1d53c}[X\u{2212}P | P] = 0 (P g\u{e9}n\u{e9}ral).");


=== Cas 1 : projection sur Y (cours). Y n'est PAS fonction de X. ===


  H(X)        = 3.433 bits


  H(X | Y)    = 2.322 bits  (variation inexpliquée par le cours)


  H(Y)        = 2.322 bits


  I(X;Y)      = H(X) - H(X|Y) = 1.111 bits  (info que le cours porte sur la note)


  Écart H(Y) − I(X;Y) = 1.211 bits


... (19 outputs left out)

  → π = 𝟙{X≥15} n'est PAS la projection L2 M = 𝔼[X|·] :


    l'orthogonalité 𝔼[X−π|π]=0 ne tient qu'au sens L2 (régression),


    pas pour une projection générale. Identité universelle :


    𝔼[X−M | M] = 0 (M = projection L2) ; pas 𝔼[X−P | P] = 0 (P général).


### Lecture du résultat

Deux scénarios, deux identités :

- **Cas 1 (Y n'est pas fonction de X)** : la règle de chaîne
  `H(X) = H(X|Y) + I(X;Y)` est vérifiée numériquement. L'écart
  `H(Y) − I(X;Y) ≈ 1.21 bits` est le **témoin** que Y porte de
  l'information sur autre chose que X — c'est précisément ce qui
  empêche d'écrire `H(X) = H(Y) + H(X|Y)`. Ici, parler de
  « projection `π(X) := Y` » n'a aucun sens, puisque Y n'est pas
  une fonction de X.
- **Cas 2 (π est fonction de X)** : la chaîne
  `H(X) = H(π(X)) + H(X | π(X))` tient, et `I(X ; π(X)) = H(π(X))`
  (et **non 0**, contrairement à ce que la cellule 8 affirmait).
  `H(π(X))` capture « ce que la projection capte » (le bit
  d'appartenance à la fibre `high` ou `low`) ; `H(X | π(X))`
  capture « ce qui reste dans les fibres » (la variation de note
  à l'intérieur d'une même fibre).

**Sémantique opérationnelle** :
- `H(π(X))` ≈ « ce que la projection capte » (la partition qu'elle définit).
- `H(X | π(X))` ≈ « ce qui reste dans les fibres » (la variation intra-fibre).
- `I(X ; π(X)) = H(π(X))` ≠ 0 en général : la projection **porte** de
  l'information sur X. L'identité `𝔼[X − π(X) | π(X)] = 0` n'est pas
  universelle — elle ne tient **que** pour la projection L2
  `M := 𝔼[X | ·]` (orthogonalité par construction de l'espérance
  conditionnelle, cf. cellule 8 invariants). Pour une projection
  générale `P = g(X)` — ex. `P = 𝟙{X ≥ 15}` du Cas 2 — les témoins
  `𝔼[X − P | P = high] ≈ 16.0` et `𝔼[X − P | P = low] ≈ 11.93`
  (cellule 9) montrent qu'elle ne tient pas, et l'indépendance
  `I(X − π(X) ; π(X)) = 0` n'est elle-même garantie que dans le cas
  gaussien. Cette lecture prolonge la cellule 9 : `π(X) = Y` n'est
  pas une *projection* au sens de la primitive 2, c'est une variable
  auxiliaire (Cas 1) ; la primitive 2 ne s'applique proprement qu'à
  un `π` qui est une fonction de X (Cas 2).

### 2.3 Test de limite — quand la primitive ne transporte plus

La règle de chaîne **H(X) = H(X|Y) + I(X;Y)** est-elle valide pour
**n'importe quelle** « projection » `Y` ?



In [7]:
// Test de limite : projection arbitraire sur un objet non-probabiliste
// On prend un texte brut et on essaie de definir H(X) comme nombre de mots uniques.
// Mais sans distribution de probabilite, ce n'est PAS de l'entropie de Shannon.
let texte = "le chat mange le poisson le chien mange la viande";
let mots: Vec<&str> = texte.split(' ').collect();
println!("Texte : '{}'", texte);
// Compteur a ordre d'insertion premier-vu (semantique Counter de Python).
let mut dist: Vec<(&str, i64)> = Vec::new();
for &m in &mots {
    if let Some(e) = dist.iter_mut().find(|e| e.0 == m) { e.1 += 1; } else { dist.push((m, 1)); }
}
println!("Mots uniques : {}, Mots totaux : {}", dist.len(), mots.len());
// H_naive = -sum(1/N * log2(1/N)) sur N = nb total mots ? NON : ce n'est pas une distribution.
// La primitive EXIGE une distribution de probabilite sur X et Y.
// Sur du texte brut, on a un COMPTAGE, pas une distribution : il faut normaliser.
// Construire une distribution : P(mot) = count / total
let n = mots.len() as i64;
let probas: Vec<(&str, f64)> = dist.iter().map(|&(m, c)| (m, c as f64 / n as f64)).collect();
let h_text = -probas.iter().filter(|(_, p)| *p > 0.0).map(|(_, p)| p * p.log2()).sum::<f64>() + 0.0;
println!("Distribution normalis\u{e9}e \u{2192} H(text) = {:.3} bits", h_text);
println!("Une fois normalis\u{e9}e, l'entropie a un sens. Mais :");
println!("  - Pas de \u{3c3}-alg\u{e8}bre canonique sur 'mots'");
println!("  - 'Y' = 'premier mot de la phrase' ? \u{2014} Y est-il une v.a. l\u{e9}gitime ?");
println!("  - Sans mesure de probabilit\u{e9}, 'projection' n'a pas de sens formel");
println!();
println!("Verdict Primitive 2 :");
println!("  D\u{e9}composition X = \u{3c0}(X) + (X|\u{3c0}(X)) exige DISTRIBUTION DE PROBABILIT\u{c9}.");
println!("  Transportable LARGE (la r\u{e8}gle de cha\u{ee}ne est universelle en th\u{e9}orie de l'information).");
println!("  Mais 'projection' et 'fibres' doivent \u{ea}tre formalis\u{e9}es en proba \u{2014}");
println!("  sinon la d\u{e9}composition est un formalisme vide.");


Texte : 'le chat mange le poisson le chien mange la viande'


Mots uniques : 7, Mots totaux : 10


Distribution normalisée → H(text) = 2.646 bits


Une fois normalisée, l'entropie a un sens. Mais :


  - Pas de σ-algèbre canonique sur 'mots'


  - 'Y' = 'premier mot de la phrase' ? — Y est-il une v.a. légitime ?


... (2 outputs left out)

  Décomposition X = π(X) + (X|π(X)) exige DISTRIBUTION DE PROBABILITÉ.


  Transportable LARGE (la règle de chaîne est universelle en théorie de l'information).


  Mais 'projection' et 'fibres' doivent être formalisées en proba —


  sinon la décomposition est un formalisme vide.


### Verdict Primitive 2

**Transportable large** : la **règle de chaîne** `H(X) = H(X|Y) + I(X;Y)` est **universelle en théorie de l'information** — elle n'exige qu'un espace probabilisé `(Ω, ℱ, ℙ)` et deux v.a. `X`, `Y`. C'est la primitive la plus transportable des trois.

**Mais** : pour qu'elle ait une **sémantique opérationnelle**, il faut que la projection `π(X) := 𝔼[X | Y]` soit **bien définie** — c'est-à-dire que `Y` soit une variable aléatoire légitime (mesurable, à valeurs dans un espace convenable). Sur du texte brut, du graphe, ou tout objet sans mesure canonique, la décomposition est un formalisme vide.

**Leçon** : la primitive est **plus transportable que Ruzsa**, mais elle exige quand même un cadre probabiliste. Le mapping « projection/fibres » vers ICT (où « fibres » = classes d'équivalence, « projection » = quotient) est **légitime** SI on construit explicitement la mesure.


### Exercice 2 — Décomposer un autre dataset en projection / fibres

Le cours décompose un dataset ML par projection sur une coordonnée puis étudie les fibres. Refaites l'exercice sur le **dataset jouet des notes** : 8 étudiants `(maths, physique)`, projection sur `maths >= 10`, fibres = les deux groupes. Mesurez pour chaque fibre : effectif, moyenne `physique`, écart-type.

**Indice** : une fibre est simplement la sous-liste des points partageant la même valeur de projection.
- **Etape 1** : écrire `projeter(points, seuil)` qui renvoie (admis, refuses).
- **Etape 2** : écrire `stats(fibre)` -> (effectif, moyenne, ecart-type).

In [8]:
// Exercice 2 — decomposition projection / fibres sur les notes (a completer)
#[allow(dead_code)]
fn projeter(points: &[(i64, i64)], seuil: i64) -> (Vec<(i64, i64)>, Vec<(i64, i64)>) {
    // Separe les points selon la projection booleenne maths >= seuil.
    (Vec::new(), Vec::new()) // stub pedagogique (regle C.1)
}
#[allow(dead_code)]
fn stats_fibre(fibre: &[(i64, i64)]) -> (usize, f64, f64) {
    // (effectif, moyenne physique, ecart-type physique) d'une fibre.
    (0, 0.0, 0.0) // stub pedagogique (regle C.1)
}
println!("Exercice a completer : decomposer [(12, 14), (8, 6), (15, 11), (9, 13), (11, 10), (6, 5), (14, 16), (10, 9)].");
println!("Attendu : comparer la dispersion par fibre a la dispersion globale.");


Exercice a completer : decomposer [(12, 14), (8, 6), (15, 11), (9, 13), (11, 10), (6, 5), (14, 16), (10, 9)].


Attendu : comparer la dispersion par fibre a la dispersion globale.


## 3. Primitive 3 — La fonctionnelle τ

### 3.1 Énoncé

Soit `S` une structure combinatoire munie d'une fonctionnelle `τ : S → ℝ⁺` qui décroît **strictement** à chaque étape d'une procédure de raffinement :

```
S₀ ⊃ S₁ ⊃ S₂ ⊃ ...
```

avec `τ(Si₊₁) < τ(Si)`. Quand `τ` **ne peut plus descendre** (`τ(Si₊₁) = τ(Si)`), la procédure s'arrête et la **structure est forcée**.

**Sémantique** : `τ` est un **analogue de fonction de Lyapunov pour la preuve**. C'est l'invariant qui garantit la terminaison.

### 3.2 Patron : descente monotone sur un problème jouet

On prend le problème « factoriser un entier `N` » avec `τ(N) = nombre de diviseurs premiers distincts`. La procédure : tester la divisibilité par 2, 3, 5, 7, 11, ... jusqu'à `√N`.


In [9]:
// Primitive 3 — τ = nombre de diviseurs premiers distincts (omega(N)), sur N = 60.
// Procédure : à chaque étape, retirer toute la valuation du plus petit facteur premier.
#[allow(dead_code)]
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
#[allow(dead_code)]
fn py_ints(v: &[u64]) -> String {
    format!("[{}]", v.iter().map(|x| x.to_string()).collect::<Vec<_>>().join(", "))
}
#[allow(dead_code)]
fn py_bools(v: &[bool]) -> String {
    format!("[{}]", v.iter().map(|&b| py_bool(b)).collect::<Vec<_>>().join(", "))
}
fn omega(n0: u64) -> u64 {
    // Nombre de diviseurs premiers distincts de N.
    let (mut n, mut count, mut p) = (n0, 0u64, 2u64);
    while p * p <= n {
        if n % p == 0 {
            count += 1;
            while n % p == 0 { n /= p; }
        }
        p += 1;
    }
    if n > 1 { count += 1; }
    count
}

// Trace : N0 = 60, on retire 2^2 -> 15, puis 3 -> 5, puis 5 -> 1.
// Chaque étape élimine un facteur premier distinct :
// ω(60)=3, ω(15)=2, ω(5)=1, ω(1)=0.
let n = 60u64;
println!("\u{3c9}({}) = {} (2\u{d7}2\u{d7}3\u{d7}5 \u{2192} facteurs premiers distincts 2, 3 et 5)", n, omega(n));

// Re-vérification : 60 = 2² × 3 × 5 → 3 facteurs premiers distincts.
let expected = {
    let mut s = std::collections::BTreeSet::new();
    s.insert(2u64); s.insert(3); s.insert(5);
    s.len() as u64
};
println!("V\u{e9}rif manuelle : 60 = 2\u{b2} \u{d7} 3 \u{d7} 5 \u{2192} \u{3c9} = {}", expected);

// La procédure de raffinement s'arrête à N=1, où ω=0.
let mut n = 60u64;
let mut steps = vec![n];
while n > 1 {
    let mut divided = false;
    for p in 2..((n as f64).sqrt() as u64 + 1) {
        if n % p == 0 {
            while n % p == 0 { n /= p; }
            steps.push(n);
            divided = true;
            break;
        }
    }
    if !divided {
        // N est premier : retirer son unique facteur premier restant.
        n = 1;
        steps.push(n);
    }
}

let omega_steps: Vec<u64> = steps.iter().map(|&s| omega(s)).collect();
let strict_decrease: Vec<bool> = (1..omega_steps.len())
    .map(|i| omega_steps[i] < omega_steps[i - 1]).collect();
println!("S\u{e9}quence de raffinement : {}", py_ints(&steps));
println!("\u{3c9} \u{e0} chaque \u{e9}tape : {}", py_ints(&omega_steps));
println!("\u{3c9} d\u{e9}cro\u{ee}t-elle strictement ? {}", py_bools(&strict_decrease));


ω(60) = 3 (2×2×3×5 → facteurs premiers distincts 2, 3 et 5)


Vérif manuelle : 60 = 2² × 3 × 5 → ω = 3


Séquence de raffinement : [60, 15, 5, 1]


ω à chaque étape : [3, 2, 1, 0]


ω décroît-elle strictement ? [True, True, True]


### Lecture du résultat

La séquence `60 → 15 → 5 → 1` montre `ω` **strictement décroissant** jusqu'à 0. La procédure **termine** parce que `ω` est un entier positif et ne peut pas descendre en dessous de 0.

**Point clé** : c'est exactement le rôle de `τ_strictly_decreases` dans ANALYSE-03 (lac `pfr`) — c'est **l'invariant qui garantit la terminaison** de la procédure de preuve. Sans cet invariant strictement décroissant, on ne pourrait pas conclure que la procédure s'arrête.


### Exercice 3 — Trouver le τ d'Euclide

Le patron du cours exige une fonctionnelle `τ` qui décroît **strictement** à chaque étape d'une procédure, sur un ordre bien fondé. Le cas positif le plus ancien : **l'algorithme d'Euclide** `(a, b) ↦ (b, a mod b)`. Le `τ` qui prouve sa terminaison est le **second argument** : `τ(a, b) = b`.

Vérifiez-le : pour plusieurs couples `(a, b)`, construisez la trajectoire des paires et la suite des valeurs de `τ` ; constatez qu'elle est **strictement décroissante** et atteint 0 — l'ordre sur ℕ est bien fondé, la procédure termine donc forcément.

**Indice** : `a mod b < b` est l'inégalité qui fait tout le travail ; la suite des seconds arguments est exactement la trajectoire de `τ`.
- **Etape 1** : écrire `trajectoire_euclide(a, b)` qui renvoie la liste des paires successives jusqu'à `(g, 0)`.
- **Etape 2** : extraire la suite des `τ` et vérifier la décroissance stricte jusqu'à 0.


In [10]:
// Exercice 3 -- trouver le tau d'Euclide : tau(a, b) = b (a completer)
#[allow(dead_code)]
fn trajectoire_euclide(a: i64, b: i64) -> Vec<(i64, i64)> {
    // Liste des paires (a, b) -> (b, a % b) -> ... jusqu'à (g, 0).
    Vec::new() // TODO etudiant (Etape 1)
}
#[allow(dead_code)]
fn suite_tau_euclide(a: i64, b: i64) -> Vec<i64> {
    // Suite des tau (seconds arguments) le long de la trajectoire.
    Vec::new() // TODO etudiant (Etape 2)
}
println!("Exercice a completer : trajectoire d'Euclide et suite des tau (ex. (252, 105), (1071, 462)).");
println!("Attendu : suite strictement decroissante jusqu'a 0 -- terminaison forcee par l'ordre sur N.");


Exercice a completer : trajectoire d'Euclide et suite des tau (ex. (252, 105), (1071, 462)).


Attendu : suite strictement decroissante jusqu'a 0 -- terminaison forcee par l'ordre sur N.


### Exercice 4 — Collatz, le cas limite : mesurer l'absence de τ

La conjecture de Collatz (`n` pair `↦ n/2` ; `n` impair `↦ 3n+1`) résiste précisément parce que **personne ne connaît de τ qui décroisse strictement le long d'une trajectoire** — sa terminaison est un problème ouvert. Mesurez cette absence : pour la trajectoire de `n = 27` (une des longues), suivez les trois candidats naturels — la **valeur** `n`, le **nombre de bits** `n.bit_length()`, et `ω(n)` (diviseurs premiers distincts, comme au §3.2) — et comptez les pas où chacun **augmente strictement**.

**Indice** : les valeurs restent < 10⁴, l'essai de division jusqu'à √v suffit pour `ω` ; un compte d'augmentations > 0 est la preuve mesurée qu'un candidat n'est pas un `τ` — l'exact contraire de la cellule du cours, où `ω` descendait de 3 à 0 sans jamais remonter.
- **Etape 1** : écrire `trajectoire_collatz(n)` et la parcourir pour `n = 27`.
- **Etape 2** : pour chaque candidat, compter les pas strictement croissants et afficher les trois comptes.


In [11]:
// Exercice 4 -- Collatz n=27 : valeur, bits et omega(n) ne decroissent pas strictement (a completer)
#[allow(dead_code)]
fn trajectoire_collatz(n: i64) -> Vec<i64> {
    // Liste des valeurs successives jusqu'à 1 (pair -> n//2, impair -> 3n+1).
    Vec::new() // TODO etudiant (Etape 1)
}
#[allow(dead_code)]
fn omega_petit(v: i64) -> usize {
    // Nombre de diviseurs premiers distincts de v (v < 10**4 : essai de division suffit).
    0 // TODO etudiant (Etape 2)
}
println!("Exercice a completer : trajectoire de n=27, compter les pas croissants de n, bit_length(n), omega(n).");
println!("Attendu : trois comptes > 0 -- aucun candidat n'est un tau strict ; la terminaison reste ouverte.");


Exercice a completer : trajectoire de n=27, compter les pas croissants de n, bit_length(n), omega(n).


Attendu : trois comptes > 0 -- aucun candidat n'est un tau strict ; la terminaison reste ouverte.


### 3.3 Test de limite — quand τ ne transporte plus

On porte `τ` sur SAT, où aucun ordre n'est donné d'avance. On construit une formule à
3 variables, on descend l'affectation `x₁, x₂, x₃`, et on **mesure la trajectoire** de
deux `τ` naturelles au lieu de supposer ce qu'elles font. Puis on regarde **2-SAT**,
où le problème, lui, *fournit* l'ordre.


In [12]:
// 3.3 — Test de limite : porter τ sur SAT, où aucun ordre n'est donné d'avance.
// On fixe x₁, x₂, x₃ dans cet ordre et on regarde ce que chaque candidat τ
// fait RÉELLEMENT le long de la descente — au lieu de le supposer.
use std::collections::{HashMap, HashSet};
#[allow(dead_code)]
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
#[allow(dead_code)]
fn py_ints(v: &[usize]) -> String {
    format!("[{}]", v.iter().map(|x| x.to_string()).collect::<Vec<_>>().join(", "))
}
#[allow(dead_code)]
fn py_bools(v: &[bool]) -> String {
    format!("[{}]", v.iter().map(|&b| py_bool(b)).collect::<Vec<_>>().join(", "))
}
fn get_assign(assign: &[(usize, bool)], v: usize) -> Option<bool> {
    assign.iter().find(|e| e.0 == v).map(|e| e.1)
}
fn clauses_non_satisfaites(clauses: &[(i64, i64, i64)], assign: &[(usize, bool)]) -> usize {
    // Clauses qu'aucun littéral déjà affecté ne rend vraie.
    clauses.iter().filter(|&&(l1, l2, l3)| {
        ![(l1), (l2), (l3)].iter().any(|&l| {
            get_assign(assign, l.unsigned_abs() as usize) == Some(l > 0)
        })
    }).count()
}

let f: [(i64, i64, i64); 5] = [(1, 2, 3), (-1, 2, -3), (1, -2, 3), (-1, -2, -3), (1, 2, -3)];
let descente: [Vec<(usize, bool)>; 4] = [
    vec![],
    vec![(1, true)],
    vec![(1, true), (2, true)],
    vec![(1, true), (2, true), (3, true)],
];

let tau_a: Vec<usize> = descente.iter().map(|a| clauses_non_satisfaites(&f, a)).collect();
let strict_a: Vec<bool> = (1..tau_a.len()).map(|i| tau_a[i] < tau_a[i - 1]).collect();
println!("candidat A : \u{3c4} = clauses non encore satisfaites");
println!("  trajectoire       : {}", py_ints(&tau_a));
println!("  d\u{e9}cro\u{ee}t strictement ? {}  \u{2192}  {}", py_bools(&strict_a), py_bool(strict_a.iter().all(|&b| b)));
println!("  Le dernier pas est PLAT : \u{3c4}_A d\u{e9}cro\u{ee}t, mais pas strictement.");
println!("  Elle ne peut donc pas fonder la terminaison.");

let tau_b: Vec<usize> = descente.iter().map(|a| 3 - a.len()).collect();
let strict_b: Vec<bool> = (1..tau_b.len()).map(|i| tau_b[i] < tau_b[i - 1]).collect();
println!("\ncandidat B : \u{3c4} = variables non encore affect\u{e9}es");
println!("  trajectoire       : {}", py_ints(&tau_b));
println!("  d\u{e9}cro\u{ee}t strictement ? {}  \u{2192}  {}", py_bools(&strict_b), py_bool(strict_b.iter().all(|&b| b)));
println!("  Elle d\u{e9}cro\u{ee}t strictement \u{2014} mais QUOI QU'IL ARRIVE : elle ne lit jamais F.");
println!("  Elle prouve que l'\u{e9}num\u{e9}ration s'arr\u{ea}te, pas que la formule est d\u{e9}cid\u{e9}e.");

// Sur 2-SAT, en revanche, le problème FOURNIT l'ordre : les composantes
// fortement connexes du graphe d'implication, et leur condensation est un DAG.
fn composantes_fortes(sommets: &[i64], succ: &HashMap<i64, Vec<i64>>) -> HashMap<i64, usize> {
    // Kosaraju — composantes fortement connexes, sans dépendance externe.
    let mut vus: HashSet<i64> = HashSet::new();
    let mut ordre: Vec<i64> = Vec::new();
    for &depart in sommets {
        if vus.contains(&depart) { continue; }
        vus.insert(depart);
        let mut pile: Vec<(i64, std::vec::IntoIter<i64>)> =
            vec![(depart, succ.get(&depart).cloned().unwrap_or_default().into_iter())];
        loop {
            let next = {
                let top = pile.last_mut().unwrap();
                let mut found = None;
                while let Some(v) = top.1.next() {
                    if !vus.contains(&v) { found = Some(v); break; }
                }
                found
            };
            if let Some(v) = next {
                vus.insert(v);
                pile.push((v, succ.get(&v).cloned().unwrap_or_default().into_iter()));
            } else {
                let (n, _) = pile.pop().unwrap();
                ordre.push(n);
                if pile.is_empty() { break; }
            }
        }
    }
    let mut pred: HashMap<i64, Vec<i64>> = HashMap::new();
    for (&a, voisins) in succ.iter() {
        for &b in voisins {
            pred.entry(b).or_default().push(a);
        }
    }
    let mut comp: HashMap<i64, usize> = HashMap::new();
    let mut vus2: HashSet<i64> = HashSet::new();
    let mut numero = 0usize;
    for &depart in ordre.iter().rev() {
        if vus2.contains(&depart) { continue; }
        vus2.insert(depart);
        let mut pile2 = vec![depart];
        let mut bloc: Vec<i64> = Vec::new();
        while let Some(n) = pile2.pop() {
            bloc.push(n);
            for &m in pred.get(&n).map(|v| v.as_slice()).unwrap_or(&[]) {
                if !vus2.contains(&m) { vus2.insert(m); pile2.push(m); }
            }
        }
        for &n in &bloc { comp.insert(n, numero); }   // un seul numéro pour TOUT le bloc
        numero += 1;
    }
    comp
}
fn deux_sat(clauses: &[(i64, i64)], nvars: usize) -> (bool, usize) {
    // (a ∨ b) ≡ (¬a → b) ∧ (¬b → a) ; satisfiable ssi jamais v et ¬v ensemble.
    let mut succ: HashMap<i64, Vec<i64>> = HashMap::new();
    for &(a, b) in clauses {
        succ.entry(-a).or_default().push(b);
        succ.entry(-b).or_default().push(a);
    }
    let mut sommets: Vec<i64> = Vec::new();
    for v in 1..=nvars {
        sommets.push(v as i64);
        sommets.push(-(v as i64));
    }
    let comp = composantes_fortes(&sommets, &succ);
    let ok = (1..=nvars).all(|v| comp.get(&(v as i64)) != comp.get(&(-(v as i64))));
    let mut nums: HashSet<usize> = HashSet::new();
    for c in comp.values() { nums.insert(*c); }
    (ok, nums.len())
}

println!("\n2-SAT : le probl\u{e8}me fournit l'ordre (condensation du graphe d'implication)");
let cas: [(&str, &[(i64, i64)], usize); 2] = [
    ("(x\u{2081} \u{2228} \u{ac}x\u{2082}) \u{2227} (\u{ac}x\u{2081} \u{2228} x\u{2082}) \u{2227} (x\u{2081} \u{2228} x\u{2082})", &[(1, -2), (-1, 2), (1, 2)], 2),
    ("(x\u{2081} \u{2228} x\u{2081}) \u{2227} (\u{ac}x\u{2081} \u{2228} \u{ac}x\u{2081})",             &[(1, 1), (-1, -1)],          1),
];
for &(libelle, clauses, nvars) in &cas {
    let (ok, n) = deux_sat(clauses, nvars);
    println!("  {:<38} \u{2192} {} composante(s), satisfiable = {}", libelle, n, py_bool(ok));
}
println!("\n  \u{3c4} n'est pas une formule qu'on transporte : c'est un patron qu'on");
println!("  CONSTRUIT \u{e0} partir d'une structure que le probl\u{e8}me doit fournir.");
println!("  2-SAT la fournit, et la d\u{e9}cision suit. SAT g\u{e9}n\u{e9}ral ne la fournit pas,");
println!("  et les deux \u{3c4} naturelles ci-dessus \u{e9}chouent chacune \u{e0} leur fa\u{e7}on.");


candidat A : τ = clauses non encore satisfaites


  trajectoire       : [5, 2, 1, 1]


  décroît strictement ? [True, True, False]  →  False


  Le dernier pas est PLAT : τ_A décroît, mais pas strictement.


  Elle ne peut donc pas fonder la terminaison.


candidat B : τ = variables non encore affectées


... (7 outputs left out)

  τ n'est pas une formule qu'on transporte : c'est un patron qu'on


  CONSTRUIT à partir d'une structure que le problème doit fournir.


  2-SAT la fournit, et la décision suit. SAT général ne la fournit pas,


  et les deux τ naturelles ci-dessus échouent chacune à leur façon.


## 4. Friction naturelle — ce qui a résisté pendant la digestion

La digestion des trois primitives n'a pas été linéaire. Trois obstacles ont rythmé le travail, et les nommer ici est le contenu réel de cette section — pas un préambule.

### 4.1 Pourquoi Ruzsa a résisté au test (a)

La cellule `[4]` ne mesure pas Ruzsa « sur son terrain ». Elle le mesure **sur le même ensemble** `{0, 1, 2, 3}`, avec **les mêmes distributions** `{0: 0.5, 1: 0.5}`, sous **deux lois de groupe différentes** — `ℤ/4ℤ` cyclique et `(ℤ/2ℤ)²` Klein. La difficulté rencontrée : `d` change de valeur quand la loi change, alors que distributions et ensemble sont fixés. Ce **n'est pas** un artefact numérique ; c'est une propriété de Ruzsa, et l'obstacle pédagogique est précisément d'**exhiber cette dépendance** sans tomber dans le piège inverse (« la valeur de `d` est invariante par changement de groupe »). Le testeur novice qui verrait `d=0` pour `ℤ/4ℤ` et `d=0` pour Klein conclurait à l'invariance — c'est exactement l'erreur que la cellule défait.

L'obstacle rencontré en digestion a été de **garder la cellule exécutable** malgré cette nuance : il a fallu calculer `H(X − Y)` sous les deux lois pour rendre la dépendance visible. Une version antérieure se contentait d'affirmer l'invariance en prose, et **tombait dans le piège inverse**. La leçon : sur Ruzsa, le test de limite n'a de valeur que **mesuré**, pas **annoncé**.

### 4.2 Pourquoi la décomposition projection/fibres cède sur le texte brut

La cellule `[8]` ne mesure pas la décomposition sur un vrai couple `(X, Y)` probabilisé. Elle **tente** la décomposition sur un texte brut où `Y` n'a pas de statut probabiliste. La difficulté rencontrée : `H(text)` après normalisation **a un sens**, mais `H(X | Y)` **n'en a pas** — il n'y a pas de σ-algèbre canonique sur « mots ». L'obstacle pédagogique est de **reconnaître cette absence** sans conclure que la décomposition est fausse : elle est **incomplète**, pas invalide. Construire une mesure rendrait la décomposition légitime, mais ce n'est pas le propos de la cellule.

L'obstacle rencontré en digestion a été de **ne pas régresser** : la tentation est grande, voyant `H(text)` fini, d'inventer un `Y` (« premier mot de la phrase » ?) qui rendrait la cellule présentable. La leçon : sur la décomposition projection/fibres, **la limite est une absence de mesure**, et il faut la nommer comme telle — pas la maquiller en pseudo-mesure pour faire passer l'exécution.

### 4.3 Pourquoi τ a cédé sur SAT, doublement

La cellule `[12]` ne mesure pas τ « en action ». Elle mesure **deux** candidats τ sur SAT 3-SAT puis 2-SAT. La difficulté rencontrée : aucune des deux τ ne fait ce qu'on espère naïvement. La première (`τ = clauses non encore satisfaites`) **descend mais pas strictement** — trajectoire `[5, 4, 1, 1]`, dernier pas plat. La seconde (`τ = variables non encore affectées`) **descend strictement** mais **ne lit jamais la formule** — elle prouve la terminaison de l'énumération, pas la décidabilité de la formule. Laquelle des deux « échoue » ? Les deux, mais **chacune à sa manière**. Le test naïf « τ décroît ⇒ ça marche » est invalide : il faut distinguer **lis** (la fonctionnelle lit-elle la structure du problème ?) et **décroît** (descend-elle bien ?).

L'obstacle rencontré en digestion a été de **ne pas tomber dans le sophisme** : il est tentant de conclure « τ_B marche, c'est la bonne ». C'est faux : τ_B prouve une chose **différente** de ce qu'on cherche. La leçon : sur τ, le test de limite doit **distinguer** terminaison d'énumération et décidabilité du problème — ces deux notions se confondent souvent dans la prose, jamais dans une mesure.


## 5. Chemin de découverte — ce qui distingue une digestion d'une reconstruction

### 5.1 Pourquoi aller vers ANALYSE-04, pas rester sur ANALYSE-03

ANALYSE-03 contient la preuve formelle PFR dans le lac `pfr` (https://github.com/teorth/pfr). L'investigation qui a mené à ANALYSE-04 est partie d'un constat : la preuve est **instructive en acte**, mais **pas en patron**. Le pattern « entropie ⇒ structure algébrique » y est **utilisé** mais pas **extrait**. Aller vers ANALYSE-04 a consisté à **isoler trois primitives candidates** dans la preuve — Ruzsa, projection/fibres, τ — et à **mesurer leur transportabilité** hors de leur terrain. Sans cette mesure, ANALYSE-04 n'aurait pas de contenu propre : il aurait été une relecture d'ANALYSE-03.

### 5.2 Essais ratés utiles

Trois essais ratés ont structuré le notebook :

- **L'essai « appliquons Ruzsa à un dataset ML »** (notes d'étudiants) a échoué parce que les notes ne sont pas des variables aléatoires sur un groupe additif. Cet échec a forcé à **déplacer le test de limite de Ruzsa vers l'algébrique** (`ℤ/4ℤ` vs Klein) plutôt que vers l'applicatif, où la limite est invisible. La cellule `[4]` est le résidu utile de cet échec.

- **L'essai « mesurons τ sur factorisation »** a échoué parce que la factorisation termine trivialement (le dernier pas de `ω` est toujours plat quand `N` est premier). Cet échec a forcé à **chercher un problème où τ n'a pas d'ordre naturel** — SAT est ce candidat. La cellule `[12]` est le résidu utile.

- **L'essai « inversons la lecture : lisons la conclusion d'abord »** (trois primitives, ranger par transportabilité) a échoué parce que la lecture de la preuve formelle ne donne **pas** l'inventaire des primitives : elle les utilise sans les nommer. Cet échec a forcé à **repartir d'ANALYSE-03, cellule par cellule** pour extraire les primitives — l'ordre du notebook est celui de la lecture, pas celui de la conclusion.

### 5.3 Dette résiduelle

Le notebook laisse trois angles non clos, nommés pour les tranches suivantes (pas pour la prochaine itération) :

- **PFR polynomial généralisé** : la preuve actuelle est bornée au cas `F₂ⁿ`. L'extension à `Z/nZ` avec `n` non premier est ouverte (cf. Gowers 2016 systematisation). Cette dette est **documentée**, pas un TODO technique : elle est hors du périmètre « primitives transportables ».

- **Décomposition projection/fibres sur ICT** : la primitive 2 est notée comme « probablement la plus pertinente » pour ICT (information mutuelle `I(X;Y)` analogue à entropie conditionnelle), mais la mesure canonique **n'est pas construite**. Cette construction appartient à un autre grain.

- **τ sur problèmes non-combinatoires** : la primitive 3 a été testée sur SAT et 2-SAT, tous deux combinatoires. Son transport à un problème continu (par exemple, terminaison d'un schéma itératif continu) n'est pas mesurée. Hors périmètre de cette digestion.

### 5.4 Transmission

Le notebook est conçu pour un lecteur ayant déjà vu ANALYSE-03. La section 5 (friction) doit être lue **après** les tests de limite, pas avant : la friction n'a de sens qu'**une fois la mesure exhibée**. La section 6.2 (essais ratés) éclaire pourquoi le notebook est ordonné comme il l'est — ce n'est pas un plan marketing, c'est l'ordre dans lequel la digestion a effectivement résisté.


### Verdict Primitive 3

**Transportable sous condition** : `τ_strictly_decreases` exige un **ordre bien fondé** sous-jacent (typiquement `ℕ` ou un ordinal). La cellule précédente montre que le porter sur SAT échoue de **deux façons distinctes** : la `τ` qui lit la formule ne décroît **pas strictement** (trajectoire plate au dernier pas), et celle qui décroît strictement **ne lit jamais la formule** — elle prouve que l'énumération s'arrête, pas que le problème est décidé. Sur **2-SAT** en revanche la structure existe (condensation du graphe d'implication) et la décision suit.

**Subtilité** : `τ` est un **patron**, pas une formule figée. Pour chaque problème, on doit **construire** la `τ` appropriée — et c'est souvent **l'étape créative** de la preuve. Dans PFR, `τ` est construit à partir de l'entropie ; dans ANALYSE-03, cellule 6, c'est l'invariant qui garantit que le blueprint termine.

**Leçon** : la primitive est **la moins transportable des trois** parce qu'elle dépend de la structure du problème. Mais quand elle s'applique, elle est **irremplaçable** — c'est elle qui dit « quand la procédure a fini ».

***

## 6. Conclusion — où les primitives cessent de valoir

### Tableau récapitulatif

| # | Primitive | Transport | Condition | Sans la condition |
|---|---|---|---|---|
| 1 | `d_ruzsa[X;Y]` | **conditionnel** | groupe additif `(G, +)` | `d` devient **négatif** : ce n'est plus une distance |
| 2 | `X = π(X) + (X \| π(X))` | **large** | distribution de probabilité | décomposition vide (projection non définie) |
| 3 | `τ_strictly_decreases` | **patron** | ordre bien-fondé | terminaison non garantie |

**Le test de limite était le livrable** : chaque primitive a été **appliquée hors de son terrain** et le notebook **exhibe** — par une cellule qui s'exécute, pas par un paragraphe qui l'annonce — l'endroit exact où elle cesse de valoir. C'est cette honnêteté qui distingue une « digestion » d'une « analogie ».

### Lien vers ANALYSE-03 et le lac `pfr`

ANALYSE-03 contient les **preuves formelles** dans le lac `pfr` (https://github.com/teorth/pfr). ANALYSE-04 ne les reproduit pas — il **extrait les patrons** et **vérifie leur portée**. Les trois primitives ci-dessus sont les **briques élémentaires** que la preuve PFR utilise pour passer de la compression à la structure algébrique.

### Vers ICT (sans mapping affine)

> PFR ne « prouve » rien pour ICT. Mais il fournit un **exemple certifié** où une quantité informationnelle **force** une structure algébrique. C'est un patron, pas une analogie : quand on dispose d'une « monnaie commune » (entropie) entre deux théories, et qu'on voit la même structure forcée par les mêmes contraintes, on a une **primitive transportable**, pas une ressemblance.

Pour ICT, la primitive la plus pertinente est **probablement la 2** (décomposition projection/fibres) : l'information mutuelle `I(X;Y)` y joue un rôle analogue à l'entropie conditionnelle. Mais on doit **construire explicitement la mesure**, pas plaquer la formule.

### Honnêteté finale

> Les trois primitives sont des **outils**, pas des **lois**. Leur transportabilité est **conditionnelle** et le test de limite fait partie du livrable. Le carnet ANALYSE-04 a nommé **où** chacune cesse de valoir — c'est le contenu pédagogique réel.

See #12214 — le carnet Python est le livrable du grain ; ce jumeau Rust est le livrable du portage #27931.
